In [ ]:
import pandas as pd
import psycopg2
from pgvector.psycopg2 import register_vector
import numpy as np
import os

In [ ]:
csv_files = [
    "../Results/guitar_feature_database.csv",
    "../Results/piano_feature_database.csv",
    "../Results/saxophone_feature_database.csv",
    "../Results/symphony_feature_database.csv",
    "../Results/violin_feature_database.csv"
]

In [ ]:
# Kết nối Database
conn = psycopg2.connect(
    dbname="music_retrieval", user="postgres", password="admin", host="localhost", port="5433"
)
register_vector(conn)
cursor = conn.cursor()

In [ ]:
# Khai báo danh sách các cột đặc trưng cho Layer 3
mfcc_cols = [f"mfcc_{i}" for i in range(1, 14)]
chroma_labels = ["C", "Csharp", "D", "Dsharp", "E", "F", "Fsharp", "G", "Gsharp", "A", "Asharp", "B"]
chroma_cols = [f"chroma_{label}" for label in chroma_labels]

total_inserted = 0

# Duyệt qua từng file CSV và đẩy vào Database
for file_path in csv_files:
    if not os.path.exists(file_path):
        print(f"Không tìm thấy file: {file_path}. Bỏ qua...")
        continue
        
    print(f"\nĐang xử lý file: {file_path}...")
    df = pd.read_csv(file_path)
    count = 0
    
    for index, row in df.iterrows():
        # Lấy thông tin Metadata
        file_name = row['file_name']
        path = row['path']
        instrument = row['instrument']
        segment_id = row['segment_id']
        
        # ----------------------------------------------------
        # GOM NHÓM THEO 3 LAYER TỪ 30 CỘT RỜI RẠC
        # ----------------------------------------------------
        
        # Layer 1: Physical / Rhythmic (2 chiều)
        v_layer1 = [
            float(row['zero_crossing_rate']), 
            float(row['tempo'])
        ]
        
        # Layer 2: Perceptual / Texture (3 chiều)
        v_layer2 = [
            float(row['spectral_centroid']), 
            float(row['spectral_bandwidth']), 
            float(row['spectral_rolloff'])
        ]
        
        # Layer 3: Identity / Voiceprint (25 chiều = 13 MFCC + 12 Chroma)
        v_layer3 = row[mfcc_cols + chroma_cols].values.astype(np.float32).tolist()
        
        # Câu lệnh SQL Insert vào 3 cột Layer tương ứng
        insert_query = """
            INSERT INTO music_segments 
            (file_name, path, instrument, segment_id, layer1_physical, layer2_perceptual, layer3_identity)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """
        cursor.execute(insert_query, (
            file_name, path, instrument, segment_id, 
            v_layer1, v_layer2, v_layer3
        ))
        count += 1
    
    # Lưu thay đổi (commit) sau mỗi file CSV hoàn tất
    conn.commit()
    total_inserted += count
    print(f"-> Đã chuyển đổi và chèn thành công {count} segments.")

In [ ]:
cursor.close()
conn.close()
print(f"=== Đã lưu xong {total_inserted} segments dưới dạng Subvector! ===")